# 02 Lidar DEM derivatives with Dask Gateway

This notebook computes fault-sensitive terrain derivatives from USGS 3DEP 1 m lidar DEMs and aggregates them to the 100 m analysis grid from notebook 01.

**Why aggregate.** Fault scarps are often only a few metres high. A 100 m DEM does not show them. The notebook computes derivatives at 1 m or 2 m and keeps sub-cell structure with statistics such as the maximum slope anomaly and the structure tensor.

**Why Dask Gateway.** Each 3DEP 1 m tile is about 10 km by 10 km, or about 400 MB. The tiles are independent, so the work spreads across Gateway workers. Each worker reads its tile directly from the USGS cloud storage and returns a small 100 m result. Only the notebook server writes to the store.

**Inputs.** `grid.json` from notebook 01. 3DEP 1 m tiles found through the USGS TNM Access API.

**Output.** `dem1m_features.zarr` under `STORE_ROOT`.

| Layer | Meaning |
|---|---|
| `elev` | Mean elevation |
| `slope_mean`, `slope_max` | Slope in degrees |
| `slope_anom_max` | Maximum of slope minus local mean slope (scarp indicator) |
| `lap_min`, `lap_max` | Laplacian extremes (paired convex and concave breaks at scarps) |
| `tpi{S}_rms` | RMS topographic position index at scale S metres |
| `lin_coherence`, `lin_sin2`, `lin_cos2` | Structure tensor coherence and strike of linear features (axial, doubled angle) |
| `elev_detrended`, `elev_detrended_slope` | Elevation minus a 5 km smooth surface, and its slope |
| `cov` | Fraction of the cell covered by lidar |

**Coverage.** 1 m lidar does not cover all of Nevada. Cells without lidar are NaN. LightGBM in notebook 03 handles NaN values. `cov` tells the model where the lidar exists.

**Annotations.** `VERIFY:` marks an untested assumption about an external service. `NOTE:` marks a design decision.

**Behavior on errors.** If Dask Gateway is not available, the notebook stops. It uses a local cluster only when you set `USE_LOCAL_CLUSTER = True`.

In [ ]:
import json
import os
import pathlib
import re
import time
import warnings

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
import rioxarray  # noqa: F401
import xarray as xr
from affine import Affine
from rasterio.enums import Resampling
from rasterio.transform import array_bounds, from_origin
from rasterio.warp import calculate_default_transform, reproject, transform_bounds
from rasterio.warp import transform as warp_transform
from rasterio.vrt import WarpedVRT
from scipy import ndimage

## 0. Configuration

In [ ]:
STORE_ROOT = os.environ.get("GEMS_STORE", str(pathlib.Path.home() / "gems"))
OUTPUT_NAME = "dem1m_features.zarr"

TNM_API = "https://tnmaccess.nationalmap.gov/api/v1/products"
DEM_DATASET = "Digital Elevation Model (DEM) 1 meter"     # VERIFY: TNM dataset name
TNM_PAGE = 500

PARAMS = {
    "decimate": 2,             # NOTE: 1 = full 1 m (about 16 GB per worker); 2 = 2 m (about 4 GB)
    "tpi_scales_m": (25.0, 100.0),
    "smooth_sigma_m": 3.0,     # smoothing before gradients, Laplacian, and structure tensor
    "slope_anom_window_m": 200.0,
    "geographic_work_res_m": 10.0,   # only used if a tile is in a geographic CRS
}

MAX_TILES = 8              # NOTE: test with a few tiles first. Set None for the full region.
USE_LOCAL_CLUSTER = False  # NOTE: explicit opt-in. There is no silent fallback.
GATEWAY_OPTIONS = {"worker_memory": 16, "worker_cores": 2}   # VERIFY: option names on your Gateway
MIN_WORKERS, MAX_WORKERS = 2, 40

GDAL_ENV = {
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.tiff",
    "GDAL_HTTP_MAX_RETRY": "5",
    "GDAL_HTTP_RETRY_DELAY": "2",
    "VSI_CACHE": "FALSE",
}

In [ ]:
fs, _ = fsspec.core.url_to_fs(STORE_ROOT)
def store_path(rel):
    return f"{STORE_ROOT.rstrip('/')}/{rel}"

with fs.open(store_path("grid.json")) as f:
    GRID = json.load(f)
H, W, RES = GRID["height"], GRID["width"], GRID["res"]

def make_template(grid):
    x = grid["xmin"] + grid["res"] * (np.arange(grid["width"]) + 0.5)
    y = grid["ymax"] - grid["res"] * (np.arange(grid["height"]) + 0.5)
    da = xr.DataArray(np.full((grid["height"], grid["width"]), np.nan, np.float32),
                      coords={"y": y, "x": x}, dims=("y", "x"))
    tr = from_origin(grid["xmin"], grid["ymax"], grid["res"], grid["res"])
    return da.rio.write_crs(grid["crs"]).rio.write_transform(tr)

template = make_template(GRID)
AOI_LONLAT = transform_bounds(GRID["crs"], "EPSG:4326", GRID["xmin"], GRID["ymax"] - H * RES,
                              GRID["xmin"] + W * RES, GRID["ymax"], densify_pts=51)
print(f"Grid {W} x {H}, AOI {tuple(round(v, 3) for v in AOI_LONLAT)}")

## 1. Find the 3DEP 1 m tiles

The TNM Access API returns one record per tile and lidar project. Where two projects cover the same tile, the notebook keeps the most recent publication.

In [ ]:
def tnm_products(bbox_lonlat, dataset, page=TNM_PAGE):
    items, offset = [], 0
    while True:
        r = requests.get(TNM_API, params={"datasets": dataset, "bbox": ",".join(f"{v:.5f}" for v in bbox_lonlat),
                                          "prodFormats": "GeoTIFF", "max": page, "offset": offset}, timeout=120)
        r.raise_for_status()
        j = r.json()                                   # VERIFY: response keys "items" and "total"
        batch = j.get("items", [])
        items += batch
        if not batch or len(items) >= int(j.get("total", len(items))):
            break
        offset += len(batch)
    return pd.DataFrame(items)

prod = tnm_products(AOI_LONLAT, DEM_DATASET)
if prod.empty:
    raise RuntimeError("TNM returned no 1 m DEM tiles for the AOI. Check DEM_DATASET and the AOI.")

# Tile key such as "11_x30y452" (UTM zone + tile index), taken from the download URL.
key = prod["downloadURL"].str.extract(r"USGS_1M_(\d+)_(x\d+y\d+)", expand=True)
prod["tile_key"] = key[0] + "_" + key[1]

def infer_tile_key(url, bbox):
    # Older files (USGS_one_meter_x28y435_<project>.tif) have no zone in the name. Tiles extend past
    # zone edges, so pick the zone in which the tile centre has the easting implied by x (10 km units).
    m = re.search(r"_x(\d+)y(\d+)_", url)
    if not m or not isinstance(bbox, dict):
        return None
    lon, lat = (bbox["minX"] + bbox["maxX"]) / 2, (bbox["minY"] + bbox["maxY"]) / 2
    target = int(m.group(1)) * 10_000 + 5_000
    z0 = int((lon + 180) // 6) + 1
    zone = min(range(z0 - 1, z0 + 2),
               key=lambda z: abs(warp_transform("EPSG:4326", f"EPSG:326{z:02d}", [lon], [lat])[0][0] - target))
    return f"{zone}_x{m.group(1)}y{m.group(2)}"

old_style = prod["tile_key"].isna()
prod.loc[old_style, "tile_key"] = [infer_tile_key(u, b) for u, b in
                                   zip(prod.loc[old_style, "downloadURL"], prod.loc[old_style, "boundingBox"])]
n_nokey = int(prod["tile_key"].isna().sum())
if n_nokey:
    warnings.warn(f"{n_nokey} records have no tile key. They are kept without de-duplication.")
    prod.loc[prod.tile_key.isna(), "tile_key"] = prod.loc[prod.tile_key.isna(), "downloadURL"]
prod["publicationDate"] = pd.to_datetime(prod.get("publicationDate"), errors="coerce")
tiles = prod.sort_values("publicationDate").groupby("tile_key").tail(1).reset_index(drop=True)
print(f"{len(prod)} records, {len(tiles)} unique tiles, about {len(tiles) * 0.4:.0f} GB to read")
if MAX_TILES:
    tiles = tiles.head(MAX_TILES)
    print(f"MAX_TILES = {MAX_TILES}: processing a test subset")

## 2. Tile functions

`tile_derivatives` is pure NumPy, so you can test it in the notebook. `aggregate_to_grid` warps each derivative into the part of the 100 m grid that the tile covers, with a resampling method that fits the statistic. `process_tile` reads one tile and calls both functions. It runs on the Gateway workers.

In [ ]:
def _odd(n):
    n = max(int(round(n)), 3)
    return n if n % 2 else n + 1

def tile_derivatives(z, res, p):
    """Terrain derivatives at native resolution. Returns a list of (name, array, resampling)."""
    valid = np.isfinite(z)
    if valid.mean() < 0.01:
        return None
    zf = np.where(valid, z, np.nanmean(z)).astype(np.float32)
    s = max(p["smooth_sigma_m"] / res, 0.5)
    zs = ndimage.gaussian_filter(zf, s)
    gr, gc = np.gradient(zs, res)
    ge, gn = gc, -gr                                   # east and north components
    slope = np.degrees(np.arctan(np.hypot(ge, gn))).astype(np.float32)
    lap = (ndimage.laplace(zs) / (res * res)).astype(np.float32)
    win = _odd(p["slope_anom_window_m"] / res)
    slope_anom = slope - ndimage.uniform_filter(slope, win)

    # Cells near nodata get unreliable derivatives. Mask them.
    reach = _odd(max(max(p["tpi_scales_m"]), p["slope_anom_window_m"]) / res + 6 * s)
    near_gap = ndimage.uniform_filter(valid.astype(np.float32), reach) < 0.999
    def m(a):
        return np.where(near_gap, np.nan, a).astype(np.float32)

    out = [("elev", np.where(valid, z, np.nan).astype(np.float32), "average"),
           ("slope_mean", m(slope), "average"), ("slope_max", m(slope), "max"),
           ("slope_anom_max", m(slope_anom), "max"),
           ("lap_min", m(lap), "min"), ("lap_max", m(lap), "max"),
           ("st_ee", m(ge * ge), "average"), ("st_nn", m(gn * gn), "average"), ("st_en", m(ge * gn), "average")]
    for sc in p["tpi_scales_m"]:
        tpi = zf - ndimage.uniform_filter(zf, _odd(sc / res))
        out.append((f"tpi{int(sc)}_ms", m(tpi * tpi), "average"))
    out.append(("cov_sum", valid.astype(np.float32), "sum"))   # NOTE: needs GDAL >= 3.1 for "sum"
    return out

def aggregate_to_grid(layers, src_transform, src_crs, grid):
    h, w = layers[0][1].shape
    left, bottom, right, top = array_bounds(h, w, src_transform)
    bx0, by0, bx1, by1 = transform_bounds(src_crs, grid["crs"], left, bottom, right, top, densify_pts=21)
    res = grid["res"]
    c0 = max(int(np.floor((bx0 - grid["xmin"]) / res)), 0)
    c1 = min(int(np.ceil((bx1 - grid["xmin"]) / res)), grid["width"])
    r0 = max(int(np.floor((grid["ymax"] - by1) / res)), 0)
    r1 = min(int(np.ceil((grid["ymax"] - by0) / res)), grid["height"])
    if c1 <= c0 or r1 <= r0:
        return None
    dst_transform = from_origin(grid["xmin"] + c0 * res, grid["ymax"] - r0 * res, res, res)
    src_res = abs(src_transform.a)
    out = {}
    for name, arr, method in layers:
        dst = np.full((r1 - r0, c1 - c0), np.nan if method != "sum" else 0.0, np.float32)
        reproject(arr, dst, src_transform=src_transform, src_crs=src_crs,
                  dst_transform=dst_transform, dst_crs=grid["crs"],
                  resampling=getattr(Resampling, method),
                  src_nodata=None if method == "sum" else np.nan,
                  dst_nodata=None if method == "sum" else np.nan)
        out[name] = dst
    out["cov"] = np.clip(out.pop("cov_sum") / (res / src_res) ** 2, 0, 1).astype(np.float32)
    return {"row0": r0, "col0": c0, "layers": out}

def process_tile(url, grid, p, gdal_env):
    t_start = time.time()
    with rasterio.Env(**gdal_env):
        with rasterio.open(url) as src:
            if src.crs is None:
                raise ValueError(f"No CRS: {url}")
            if src.crs.is_geographic:
                tr, w, h = calculate_default_transform(src.crs, grid["crs"], src.width, src.height,
                                                       *src.bounds, resolution=p["geographic_work_res_m"])
                with WarpedVRT(src, crs=grid["crs"], transform=tr, width=w, height=h,
                               resampling=Resampling.bilinear) as vrt:
                    z = vrt.read(1, masked=True).astype(np.float32).filled(np.nan)
                    transform, crs = tr, vrt.crs
            else:
                f = int(p["decimate"])
                oh, ow = src.height // f, src.width // f
                # Decimated reads use COG overviews when they exist, which cuts the download.
                z = src.read(1, out_shape=(oh, ow), masked=True,
                             resampling=Resampling.average if f > 1 else Resampling.nearest)
                z = z.astype(np.float32).filled(np.nan)
                transform = src.transform * Affine.scale(src.width / ow, src.height / oh)
                crs = src.crs
    lyr = tile_derivatives(z, abs(transform.a), p)
    if lyr is None:
        return None
    res = aggregate_to_grid(lyr, transform, crs, grid)
    if res is not None:
        res["url"], res["seconds"] = url, time.time() - t_start
    return res

## 3. Test one tile on the notebook server

Run one tile before you start the cluster. This checks remote access to the 3DEP bucket and the memory use for your `decimate` value.

In [ ]:
test = process_tile(tiles.downloadURL.iloc[0], GRID, PARAMS, GDAL_ENV)
if test is None:
    raise RuntimeError("The test tile returned no data. Try another tile.")
print(f"{test['seconds']:.0f} s, window {test['layers']['cov'].shape} at row {test['row0']}, col {test['col0']}")
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, n in zip(axes, ["elev", "slope_anom_max", "cov"]):
    ax.imshow(test["layers"][n]); ax.set_title(n); ax.set_axis_off()

## 4. Start the Dask cluster

The cell stops with an error if Dask Gateway is not reachable. `client.get_versions(check=True)` stops if the worker image differs from the notebook image. A mismatch causes errors that are hard to diagnose later.

In [ ]:
from dask.distributed import as_completed

if USE_LOCAL_CLUSTER:
    from dask.distributed import Client, LocalCluster
    cluster = LocalCluster(n_workers=2, threads_per_worker=1, memory_limit="8GB")
    client = Client(cluster)
    print("LOCAL cluster (explicit opt-in):", client.dashboard_link)
else:
    try:
        from dask_gateway import Gateway
        gateway = Gateway()
        options = gateway.cluster_options()
    except Exception as e:
        raise RuntimeError("Dask Gateway is not reachable from this server. Check the GeoLab "
                           "profile, or set USE_LOCAL_CLUSTER = True to run locally.") from e
    for k, v in GATEWAY_OPTIONS.items():
        if k in options:
            options[k] = v
        else:
            warnings.warn(f"Gateway option {k!r} is not offered. Offered: {list(options)}")
    cluster = gateway.new_cluster(options)
    cluster.adapt(minimum=MIN_WORKERS, maximum=MAX_WORKERS)
    client = cluster.get_client()
    print("Gateway cluster:", cluster.dashboard_link)

client.wait_for_workers(1, timeout=600)
client.get_versions(check=True)

## 5. Process all tiles and build the mosaic

Tiles overlap slightly at their edges. Where two results cover the same 100 m cell, the mosaic keeps the value from the tile with the higher lidar coverage for that cell. Failed tiles are listed at the end. They do not stop the run.

In [ ]:
names = list(test["layers"].keys())
mosaic = {n: np.full((H, W), np.nan, np.float32) for n in names}
best = np.zeros((H, W), np.float32)

def add_to_mosaic(res):
    r0, c0 = res["row0"], res["col0"]
    L = res["layers"]
    h, w = L["cov"].shape
    sl = (slice(r0, r0 + h), slice(c0, c0 + w))
    cov = np.nan_to_num(L["cov"])
    take = cov > best[sl]
    best[sl][take] = cov[take]
    for n in names:
        mosaic[n][sl][take] = L[n][take]

futures = client.map(process_tile, list(tiles.downloadURL), grid=GRID, p=PARAMS,
                     gdal_env=GDAL_ENV, retries=2, pure=False)
failed, done = [], 0
for fut in as_completed(futures):
    try:
        res = fut.result()
    except Exception as e:
        failed.append((fut.key, repr(e)))
        continue
    finally:
        fut.release()
    if res is not None:
        add_to_mosaic(res)
    done += 1
    if done % 25 == 0:
        print(f"{done}/{len(futures)} tiles")
print(f"Finished: {done} tiles, {len(failed)} failed")
for k, e in failed[:20]:
    print("  FAILED", k, e)

## 6. Derived layers and output

In [ ]:
def nan_gaussian(a, sigma):
    m = np.isfinite(a)
    num = ndimage.gaussian_filter(np.where(m, a, 0).astype(np.float32), sigma)
    den = ndimage.gaussian_filter(m.astype(np.float32), sigma)
    with np.errstate(invalid="ignore", divide="ignore"):
        out = num / den
    out[den < 1e-3] = np.nan
    return out

def structure_tensor_features(see, snn, sen, eps=1e-12):
    """Coherence (0..1) and doubled-angle strike of linear features.

    The dominant gradient direction is theta = 0.5 * atan2(2 sen, see - snn), from east.
    Linear features strike at theta + 90 degrees, so the doubled angle changes sign.
    """
    coh = np.sqrt((see - snn) ** 2 + 4 * sen ** 2) / (see + snn + eps)
    two_theta = np.arctan2(2 * sen, see - snn)
    return coh.astype(np.float32), (-np.sin(two_theta)).astype(np.float32), (-np.cos(two_theta)).astype(np.float32)

In [ ]:
out = {}
for n in ["elev", "slope_mean", "slope_max", "slope_anom_max", "lap_min", "lap_max", "cov"]:
    out[n] = mosaic[n]
for sc in PARAMS["tpi_scales_m"]:
    out[f"tpi{int(sc)}_rms"] = np.sqrt(mosaic[f"tpi{int(sc)}_ms"])
out["lin_coherence"], out["lin_sin2"], out["lin_cos2"] = structure_tensor_features(
    mosaic["st_ee"], mosaic["st_nn"], mosaic["st_en"])

# NOTE: 5 km detrending removes basin-scale relief and keeps range-front steps.
out["elev_detrended"] = mosaic["elev"] - nan_gaussian(mosaic["elev"], 5000.0 / RES)
gr, gc = np.gradient(out["elev_detrended"], RES)
out["elev_detrended_slope"] = np.degrees(np.arctan(np.hypot(gr, gc))).astype(np.float32)

dem = xr.Dataset({f"dem_{k}": (("y", "x"), v.astype(np.float32)) for k, v in out.items()},
                 coords={"y": template.y, "x": template.x}).rio.write_crs(GRID["crs"])
dem.attrs.update({"grid": json.dumps(GRID), "params": json.dumps(PARAMS, default=list),
                  "n_tiles": int(done), "n_failed": len(failed)})
dem.chunk({"y": 1024, "x": 1024}).to_zarr(store_path(OUTPUT_NAME), mode="w", consolidated=True)
print(f"Wrote {OUTPUT_NAME}: lidar covers {np.nanmean(out['cov'] > 0.5):.1%} of the grid")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for ax, n in zip(axes, ["dem_slope_anom_max", "dem_lin_coherence", "dem_elev_detrended"]):
    v = dem[n].values
    lo, hi = np.nanpercentile(v, [2, 98])
    ax.imshow(v, vmin=lo, vmax=hi); ax.set_title(n); ax.set_axis_off()

In [ ]:
client.close()
cluster.close()   # NOTE: shut down the Gateway cluster so that idle workers do not keep running